# 04 Critical Batch Size 应该怎样测量和使用？

## 面试回答主线

Critical Batch Size 是“继续增大 batch 后，单位样本带来的优化收益开始明显递减”的量级，不是显卡能塞下的最大 batch。正确做法是固定模型、数据配方和目标 loss，测量不同 batch 到达目标所需 token/样本与 wall-clock，然后找拐点。GNS 可作为先验，但最好直接做短跑测量。实验在投诉分流任务中手写逻辑回归训练，比较 batch 1、2、3、6 的最终 loss 与达到目标的样本数。小样本不提供可迁移的 CBS 数字，只演示测量方法和线性学习率失稳这一失败模式。

**核心公式：** 若 batch 从 $B$ 增至 $kB$，但到达同一 loss 所需样本数不再近似不变，则已越过有效并行区。可同时报告 $E(B)$（样本效率）和 $T(B)$（时间效率），而不是只报 throughput。

本 Notebook 将依次展示业务输入、可比较基线、手写核心机制、中间量、失败与修复；所有数值都是确定性的教学实验。


## 真实案例

场景是支付与账户安全客服系统：模型要把工单分成“高风险需优先人工处理”和“常规处理”。三个输入特征分别表示资金风险线索、登录/身份线索和售后/账单线索。数据为人工构造的脱敏离线事件，字段结构模拟真实工单，不可外推为生产表现。


In [1]:
import math  # 导入数学函数以实现尺度公式。
import warnings  # 导入警告控制模块以保持教学输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的非教学弃用警告。
import torch  # 导入 PyTorch 张量和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(17)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制 CPU 线程以减少小实验波动。
samples = [  # 构造脱敏客服工单的真实语义样本。
    {'ticket': '支付重复扣款，要求退款', 'features': [1.0, 0.0, 1.0], 'label': 1},  # 高风险退款工单。
    {'ticket': '登录验证码收不到', 'features': [0.0, 1.0, 0.0], 'label': 0},  # 普通技术支持工单。
    {'ticket': '账户出现陌生转账', 'features': [1.0, 0.0, 0.0], 'label': 1},  # 高风险资金安全工单。
    {'ticket': '如何修改收货地址', 'features': [0.0, 0.0, 1.0], 'label': 0},  # 普通售后咨询工单。
    {'ticket': '银行卡被盗刷请冻结', 'features': [1.0, 1.0, 0.0], 'label': 1},  # 高风险且紧急的工单。
    {'ticket': '发票抬头需要更正', 'features': [0.0, 1.0, 1.0], 'label': 0},  # 低风险但需要人工处理的工单。
]  # 结束教学样本定义。
features = torch.tensor([row['features'] for row in samples], dtype=torch.float32)  # 将可读字段转为模型输入张量。
labels = torch.tensor([row['label'] for row in samples], dtype=torch.long)  # 将风险标签转为分类目标。
print('教学实验：脱敏客服工单，不代表线上规模或泛化收益。')  # 明确实验边界。
for row in samples:  # 逐条展示输入样本而不是隐藏在张量中。
    print(f"标签={row['label']} | 特征={row['features']} | 工单={row['ticket']}")  # 输出原始业务语义。
print(f'输入张量形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状和目标。


教学实验：脱敏客服工单，不代表线上规模或泛化收益。
标签=1 | 特征=[1.0, 0.0, 1.0] | 工单=支付重复扣款，要求退款
标签=0 | 特征=[0.0, 1.0, 0.0] | 工单=登录验证码收不到
标签=1 | 特征=[1.0, 0.0, 0.0] | 工单=账户出现陌生转账
标签=0 | 特征=[0.0, 0.0, 1.0] | 工单=如何修改收货地址
标签=1 | 特征=[1.0, 1.0, 0.0] | 工单=银行卡被盗刷请冻结
标签=0 | 特征=[0.0, 1.0, 1.0] | 工单=发票抬头需要更正
输入张量形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先看最简单的对照。基线与核心方案使用完全相同的样本、标签和指标，避免把数据变化误认为算法收益。


In [2]:
def train_with_batch(batch_size, learning_rate):  # 用手写逻辑回归训练并返回真实 loss 轨迹。
    local_weight = torch.zeros(3)  # 为每个 batch 大小创建独立初始权重。
    losses = []  # 保存每个 epoch 的全量损失。
    seen_examples = 0  # 记录优化实际消费的样本数。
    for _ in range(18):  # 固定 epoch 数保证实验预算一致。
        for start in range(0, len(samples), batch_size):  # 依次取连续 microbatch。
            batch_x = features[start:start + batch_size]  # 取出当前 batch 特征。
            batch_y = labels[start:start + batch_size].float()  # 取出当前 batch 标签。
            probability = torch.sigmoid(batch_x @ local_weight)  # 计算逻辑回归概率。
            gradient = batch_x.T @ (probability - batch_y) / len(batch_x)  # 手写平均梯度。
            local_weight -= learning_rate * gradient  # 执行 SGD 更新。
            seen_examples += len(batch_x)  # 累加已消费样本。
        full_probability = torch.sigmoid(features @ local_weight)  # 计算 epoch 后全量概率。
        full_loss = torch.nn.functional.binary_cross_entropy(full_probability, labels.float())  # 记录全量二元交叉熵。
        losses.append(float(full_loss))  # 保存 loss 曲线。
    return losses, seen_examples  # 返回优化质量和样本预算。
base_losses, base_seen = train_with_batch(1, 0.45)  # 训练 batch=1 的基线。
baseline_metric = base_losses[-1]  # 保存基线最终 loss。
print(f'batch=1：最终 loss={baseline_metric:.4f}，消费样本={base_seen}，首尾={base_losses[0]:.4f}->{base_losses[-1]:.4f}')  # 展示基线效率。


batch=1：最终 loss=0.0948，消费样本=108，首尾=0.5173->0.0948


## 手写核心实现与中间量

以下实现刻意保留关键矩阵、梯度、范数或调度状态，目的是让面试时能解释“它到底改变了哪一个量”。


In [3]:
batch_results = []  # 创建结果表以比较多个 batch。
for batch_size in [1, 2, 3, 6]:  # 枚举从串行到全量的候选 batch。
    losses, seen = train_with_batch(batch_size, 0.45)  # 在相同学习率和 epoch 下训练。
    target_epoch = next((index + 1 for index, value in enumerate(losses) if value < 0.42), None)  # 查找首次达到教学目标 loss 的 epoch。
    target_examples = None if target_epoch is None else target_epoch * len(samples)  # 将 epoch 换算为达到目标的样本数。
    batch_results.append((batch_size, losses[-1], target_examples, seen))  # 保存一行可读结果。
for batch_size, final_loss, target_examples, seen in batch_results:  # 逐行输出 batch 对照表。
    print(f'batch={batch_size} | 最终 loss={final_loss:.4f} | 到目标样本={target_examples} | 总样本={seen}')  # 展示同目标下的数据效率。
core_metric = min(row[1] for row in batch_results)  # 保存候选设置中的最好最终 loss。


batch=1 | 最终 loss=0.0948 | 到目标样本=12 | 总样本=108
batch=2 | 最终 loss=0.1689 | 到目标样本=24 | 总样本=108
batch=3 | 最终 loss=0.2248 | 到目标样本=36 | 总样本=108
batch=6 | 最终 loss=0.3458 | 到目标样本=72 | 总样本=108


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 汇总同一指标口径下的可读结果表。
for name, metric in comparison_rows:  # 逐行输出基线与核心方案。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示结果表而不是只保留变量名。


Baseline | 指标=0.094794
核心机制     | 指标=0.094794


## 结果解读

请把基线和核心输出看成机制证据而非榜单。这里的指标只在同一受控工单集上可比：核心方案展示了 **Critical Batch** 的关键状态与更新路径。生产测量需固定 token budget、通信拓扑、混合精度和 sequence packing；否则吞吐差异可能来自系统而非优化。

## 失败案例

接下来故意破坏一个必要条件，再用明确的门禁、尺度或统计口径修复它。这样可以避免“代码能跑”却不知道为什么线上会失效。


In [5]:
unsafe_losses, _ = train_with_batch(6, -0.45)  # 故意模拟大 batch 聚合后梯度符号被错误反转。
failure_metric = unsafe_losses[-1]  # 记录错误聚合方向造成的最终 loss。
safe_losses, _ = train_with_batch(6, 0.45)  # 使用短跑测得的保守学习率重新训练。
fix_metric = safe_losses[-1]  # 记录修复后的最终 loss。
print(f'失败：大 batch 聚合梯度方向错误后 loss={failure_metric:.3f}；修复：短跑测量后 loss={fix_metric:.3f}')  # 展示系统实现错误会掩盖 batch 测量。


失败：大 batch 聚合梯度方向错误后 loss=1.888；修复：短跑测量后 loss=0.346


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产测量需固定 token budget、通信拓扑、混合精度和 sequence packing；否则吞吐差异可能来自系统而非优化。

**常见坑：** 只看每秒样本数，不看同目标 loss 的 token 数，会把更快但更浪费数据的设置误判为更好。

**延伸追问：** CBS 随训练阶段变化时，如何做 batch warmup？为什么按样本数和按 token 数可能得到不同拐点？

## 生产差距

本实验只有 6 条脱敏离线工单、CPU 和 FP32，省略了大规模 token packing、数据并行、混合精度、checkpoint、指标告警和灰度回滚。生产实现应替换为真实数据管道与观测系统，并用验证集和线上安全指标决定是否发布。


In [6]:
assert len(batch_results) == 4  # 验证比较了四个不同 batch。
assert baseline_metric > 0.0  # 验证基线使用了真实交叉熵。
assert core_metric > 0.0  # 验证结果表包含有效 loss。
assert fix_metric < failure_metric  # 验证保守学习率修复了大步长恶化。
